# YouTube transcripts in Colab, without RequestBlocked

YouTube blocks most cloud IPs, including Colab's, so youtube-transcript-api often fails here with `RequestBlocked` or `IpBlocked`.

[youtube-transcript-cloud](https://github.com/retracn/youtube-transcript-cloud) keeps youtube-transcript-api's interface, so the code below is ordinary youtube-transcript-api code. The only change is the import. Each request runs on a hosted transcript API that deals with YouTube's blocks for you.

## 1. Install

In [ ]:
!pip install -q git+https://github.com/retracn/youtube-transcript-cloud

## 2. Add your Apify token

youtube-transcript-cloud fetches through a hosted transcript API on Apify. Create a [free Apify account](https://console.apify.com/sign-up) (the free plan's $5 monthly credit covers about 3,300 transcripts) and copy your token from [Settings → API & Integrations](https://console.apify.com/settings/integrations).

In Colab, click the key icon in the left sidebar, add a secret named `APIFY_TOKEN` and turn on notebook access. The cell below reads it, or asks for the token if you skip that.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["APIFY_TOKEN"] = userdata.get("APIFY_TOKEN")
except Exception:
    pass
if not os.environ.get("APIFY_TOKEN"):
    os.environ["APIFY_TOKEN"] = getpass("Apify API token: ")

## 3. Fetch a transcript

In [ ]:
from youtube_transcript_cloud import YouTubeTranscriptApi   # instead of: from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch("dQw4w9WgXcQ", languages=["en"])

print(transcript.language, "| auto-generated:", transcript.is_generated, "|", len(transcript), "snippets")
for snippet in transcript[:5]:
    print(f"[{snippet.start:6.1f}s] {snippet.text}")

## 4. Plain text, or subtitle files

In [ ]:
from youtube_transcript_cloud.formatters import TextFormatter, SRTFormatter

text = TextFormatter().format_transcript(transcript)
print(text[:500])

with open("transcript.srt", "w") as f:
    f.write(SRTFormatter().format_transcript(transcript))

## 5. Many videos in one request

`fetch_many` runs one request for a list of videos, which is faster and cheaper than a loop. Each value is a transcript, or the error for that video.

In [ ]:
results = ytt_api.fetch_many(["dQw4w9WgXcQ", "jNQXAC9IVRw"], languages=["en"])
for video_id, result in results.items():
    print(video_id, "->", f"{len(result)} snippets" if not isinstance(result, Exception) else repr(result))

## Cost

$1.50 per 1,000 transcripts on your Apify account; videos without captions are free.

More: [youtube-transcript-cloud on GitHub](https://github.com/retracn/youtube-transcript-cloud) · [the YouTube Transcript Scraper it runs on](https://apify.com/automationnation/youtube-transcript-scraper)